This function enables patterns generated via Ernould's method to be written to a .up2 file to enable processing by ATEX 

In [1]:
import numpy as np
import os
import sys

import ErnouldsMethod
#import write_up2
import conversions
import write_up2 as write_up2

In [2]:

detector_shape = (1200,1200)
pattern_center = np.array([10, 10, 16*1000/20])

# Going From H to e and w 

In [3]:
def convertH2strain(h, pattern_center):
    print("H:")
    print(",".join(f"{x:.6g}" for x in h))

    F_vals = conversions.h2F(h, pattern_center)

    print("F_vals:")
    print("\t".join(f"{x:.6g}" for x in F_vals.flatten()))

    
    epsilon, omega = conversions.F2strain(F_vals, small_strain= False)
    print("Epsilon: ")
    print("\t".join(f"{x:.6g}" for x in epsilon.flatten()))

    w13 = omega[0, 2]
    w21 = omega[1, 0]
    w32 = omega[2, 1]

    #convert the rotation components to degrees
    w13 = np.degrees(w13)
    w21 = np.degrees(w21)
    w32 = np.degrees(w32)

    print('w13: ', w13)
    print('w21: ', w21)
    print('w32: ', w32)
    return epsilon, w13, w21, w32

# Going from Strain to H


In [4]:
def convertStrain2H(e, w13, w21, w32, pattern_center):
    e = np.array(e, dtype=np.float32)
    w = np.array([w32, w13, w21], dtype=np.float32)

    print ("Epsilon: ", e.flatten())
    print ("Omega: ", w.flatten())
    Fe_val = ErnouldsMethod.determineF(e, w)
    Fe_val = Fe_val/Fe_val[2, 2] #normalize Fe_val by F33
    print("Fe_val: ", Fe_val.flatten())
    h = conversions.F2h(Fe_val, pattern_center) 
    print("h: ", h.flatten())
    return h

# Main Code

In [5]:
e = np.array([[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]], dtype=np.float32)
w = np.array([0.4, 0.0, 0.0], dtype=np.float32) #w13 = 0.1, w21 = 0.1, w32 = 0.1 in degrees

h = convertStrain2H(e, w[0], w[1], w[2], pattern_center)

Epsilon:  [0. 0. 0. 0. 0. 0. 0. 0. 0.]
Omega:  [0.  0.4 0. ]
Fe_val:  [ 1.          0.          0.00698143  0.          1.00002438  0.
 -0.00698143  0.          1.        ]
h:  [ 1.74550991e-04  0.00000000e+00  5.58650447e+00  8.72754956e-05
  1.11656517e-04  1.11656517e-03 -8.72754956e-06  0.00000000e+00]


In [6]:
h11 = 0.0
h12 = 0.0
h13 = 0.0
h21 = 0.0
h22 = 0.0
h23 = 0.3
h31 = 0.0
h32 = 0.0


hvals = np.array([h11, h12, h13, h21, h22, h23, h31, h32], dtype=np.float32) # input h values 
epsilon, w13, w21, w32 = convertH2strain(hvals, pattern_center)

#print each of the results in a single line, separated by commas
print("Epsilon: ", ",".join(f"{x:.6g}" for x in epsilon.flatten()))
print(f"w13: {w13:.6g}, w21: {w21:.6g}, w32: {w32:.6g}")

H:
0,0,0,0,0,0.3,0,0
F_vals:
1	0	0	0	1	0.000375	0	0	1
Epsilon: 
1.75781e-08	0	0	0	7.03125e-08	0.0001875	0	0.0001875	0
w13:  0.0
w21:  0.0
w32:  -0.010742958959692626
Epsilon:  1.75781e-08,0,0,0,7.03125e-08,0.0001875,0,0.0001875,0
w13: 0, w21: 0, w32: -0.010743


In [7]:
h23_values = np.arange(0, 3, 0.25) #vary h23 from 0 to 3 in steps of 0.25
print("h23 values: ", h23_values)
w13_vals = np.zeros(len(h23_values))
w21_vals = np.zeros(len(h23_values))
w32_vals = np.zeros(len(h23_values))
epsilon_vals = np.zeros((len(h23_values), 3, 3))

for i, h23 in enumerate(h23_values):
    hvals = np.array([h11, h12, h13, h21, h22, h23, h31, h32], dtype=np.float32) # input h values 
    print(f"\nFor h23 = {h23:.2f}:")
    epsilon, w13, w21, w32 = convertH2strain(hvals, pattern_center)
    w13_vals[i] = w13
    w21_vals[i] = w21
    w32_vals[i] = w32
    epsilon_vals[i] = epsilon

h23 values:  [0.   0.25 0.5  0.75 1.   1.25 1.5  1.75 2.   2.25 2.5  2.75]

For h23 = 0.00:
H:
0,0,0,0,0,0,0,0
F_vals:
1	0	0	0	1	0	0	0	1
Epsilon: 
0	0	0	0	0	0	0	0	0
w13:  0.0
w21:  0.0
w32:  0.0

For h23 = 0.25:
H:
0,0,0,0,0,0.25,0,0
F_vals:
1	0	0	0	1	0.0003125	0	0	1
Epsilon: 
1.2207e-08	0	0	0	4.88281e-08	0.00015625	0	0.00015625	0
w13:  0.0
w21:  0.0
w32:  -0.008952465476064332

For h23 = 0.50:
H:
0,0,0,0,0,0.5,0,0
F_vals:
1	0	0	0	1	0.000625	0	0	1
Epsilon: 
4.88281e-08	0	0	0	1.95312e-07	0.0003125	0	0.0003125	0
w13:  0.0
w21:  0.0
w32:  -0.017904930515001072

For h23 = 0.75:
H:
0,0,0,0,0,0.75,0,0
F_vals:
1	0	0	0	1	0.0009375	0	0	1
Epsilon: 
1.09863e-07	0	0	0	4.39453e-07	0.00046875	0	0.00046875	0
w13:  0.0
w21:  0.0
w32:  -0.026857394679663447

For h23 = 1.00:
H:
0,0,0,0,0,1,0,0
F_vals:
1	0	0	0	1	0.00125	0	0	1
Epsilon: 
1.95312e-07	0	0	0	7.8125e-07	0.000625	0	0.000625	0
w13:  0.0
w21:  0.0
w32:  -0.03580985753293308

For h23 = 1.25:
H:
0,0,0,0,0,1.25,0,0
F_vals:
1	0	0	0	1	0.0015625	0	0	1


In [8]:
#save the results to a .csv file
np.savetxt("h23_vs_rotation.csv", np.column_stack((h23_values, w13_vals, w21_vals, w32_vals, epsilon_vals.reshape(len(h23_values), 9))), delimiter=",", header="h23,w13,w21,w32,epsilon_11,epsilon_12,epsilon_13,epsilon_21,epsilon_22,epsilon_23,epsilon_31,epsilon_32,epsilon_33", comments="")